# Noisy Image Recovery Using Spatial Filtering

Digital Image Processing Mini Project

**Goal:** Add Gaussian and Salt-and-Pepper noise to an image, recover the image using Mean and Median filters, and compare the results using MSE, PSNR, and SSIM.

## 1. Project Workflow

Original Image → Add Noise → Apply Filters → Compare Results → Calculate MSE, PSNR and SSIM

In [ ]:
# Install required libraries if needed
# Run this cell only if the packages are not already installed.
!pip install opencv-python numpy matplotlib scikit-image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
from pathlib import Path

print('Libraries imported successfully.')

## 2. Upload the Input Image

In Google Colab, run the next cell and select any JPG/PNG image from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No image was uploaded.")

IMAGE_PATH = next(iter(uploaded))

original_bgr = cv2.imread(IMAGE_PATH)

if original_bgr is None:
    raise ValueError("The uploaded file could not be read as an image.")

original = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
original = cv2.resize(original, (600, 400))

plt.figure(figsize=(8, 5))
plt.imshow(original)
plt.title("Original Image")
plt.axis("off");

## 3. Convert Image to Grayscale

In [ ]:
gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(8, 5))
plt.imshow(gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off');

## 4. Add Gaussian Noise

In [ ]:
def add_gaussian_noise(image, mean=0, sigma=25):
    noise = np.random.normal(mean, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

np.random.seed(42)
gaussian_noisy = add_gaussian_noise(gray, sigma=25)

plt.figure(figsize=(8, 5))
plt.imshow(gaussian_noisy, cmap='gray')
plt.title('Image with Gaussian Noise')
plt.axis('off');

## 5. Add Salt-and-Pepper Noise

In [ ]:
def add_salt_pepper_noise(image, amount=0.05):
    noisy = image.copy()
    total_pixels = image.size
    num_salt = int(total_pixels * amount / 2)
    num_pepper = int(total_pixels * amount / 2)

    # Salt pixels
    coords = tuple(np.random.randint(0, dim, num_salt) for dim in image.shape)
    noisy[coords] = 255

    # Pepper pixels
    coords = tuple(np.random.randint(0, dim, num_pepper) for dim in image.shape)
    noisy[coords] = 0

    return noisy

np.random.seed(42)
sp_noisy = add_salt_pepper_noise(gray, amount=0.05)

plt.figure(figsize=(8, 5))
plt.imshow(sp_noisy, cmap='gray')
plt.title('Image with Salt-and-Pepper Noise')
plt.axis('off');

## 6. Apply Mean and Median Filters

In [ ]:
# Mean filtering
gaussian_mean = cv2.blur(gaussian_noisy, (5, 5))
sp_mean = cv2.blur(sp_noisy, (5, 5))

# Median filtering
gaussian_median = cv2.medianBlur(gaussian_noisy, 5)
sp_median = cv2.medianBlur(sp_noisy, 5)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
images = [gray, gaussian_noisy, gaussian_median, sp_noisy, sp_mean, sp_median]
titles = [
    'Original', 'Gaussian Noise', 'Gaussian + Median Filter',
    'Salt-and-Pepper Noise', 'S&P + Mean Filter', 'S&P + Median Filter'
]

for ax, img, title in zip(axes.ravel(), images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout();

## 7. Calculate MSE, PSNR and SSIM

In [ ]:
def evaluate(reference, result):
    mse = mean_squared_error(reference, result)
    psnr = peak_signal_noise_ratio(reference, result, data_range=255)
    ssim = structural_similarity(reference, result, data_range=255)
    return mse, psnr, ssim

results = {
    'Gaussian Noise': evaluate(gray, gaussian_noisy),
    'Gaussian + Mean Filter': evaluate(gray, gaussian_mean),
    'Gaussian + Median Filter': evaluate(gray, gaussian_median),
    'Salt-Pepper Noise': evaluate(gray, sp_noisy),
    'S&P + Mean Filter': evaluate(gray, sp_mean),
    'S&P + Median Filter': evaluate(gray, sp_median),
}

print(f"{'Method':35} {'MSE':>12} {'PSNR (dB)':>12} {'SSIM':>12}")
print('-' * 75)
for method, (mse, psnr, ssim) in results.items():
    print(f'{method:35} {mse:12.2f} {psnr:12.2f} {ssim:12.4f}')

## 8. Visual Comparison

The following comparison helps identify which filtering method produces the clearest recovered image.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(sp_noisy, cmap='gray')
axes[1].set_title('Noisy Image')
axes[2].imshow(sp_median, cmap='gray')
axes[2].set_title('Recovered with Median Filter')
for ax in axes:
    ax.axis('off')
plt.tight_layout();

## 9. Save the Results

The output images are saved so they can be included in the project report.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

cv2.imwrite(str(output_dir / 'original.png'), gray)
cv2.imwrite(str(output_dir / 'gaussian_noisy.png'), gaussian_noisy)
cv2.imwrite(str(output_dir / 'gaussian_mean_filtered.png'), gaussian_mean)
cv2.imwrite(str(output_dir / 'gaussian_median_filtered.png'), gaussian_median)
cv2.imwrite(str(output_dir / 'salt_pepper_noisy.png'), sp_noisy)
cv2.imwrite(str(output_dir / 'salt_pepper_mean_filtered.png'), sp_mean)
cv2.imwrite(str(output_dir / 'salt_pepper_median_filtered.png'), sp_median)

print('Results saved in the outputs/ folder.')

## 10. Conclusion

This project demonstrates digital image restoration using spatial filtering. Gaussian and salt-and-pepper noise are introduced into an image, followed by mean and median filtering. MSE, PSNR and SSIM are used to quantitatively compare the restored images with the original image. The results can be discussed in terms of noise removal, detail preservation and limitations of each filter.

## 11. Viva Points

- **Why add noise?** To simulate image degradation and test restoration techniques.
- **Why median filtering?** It is particularly useful for impulse/salt-and-pepper noise.
- **What is MSE?** Mean Squared Error measures the average squared difference between images; lower is generally better.
- **What is PSNR?** Peak Signal-to-Noise Ratio measures reconstruction quality; higher is generally better.
- **What is SSIM?** Structural Similarity compares structural information between images; values closer to 1 indicate greater similarity.
- **Main DIP components:** noise modeling, spatial filtering and quantitative image-quality evaluation.